In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [35]:
ROOT_DIR = "/kaggle/input/chest-xray-pneumonia/chest_xray"
os.listdir(ROOT_DIR)

['chest_xray', '__MACOSX', 'val', 'test', 'train']

In [36]:
import os  # for file/directory operations
import cv2  # computer vision library for image processing
import numpy as np  # array operations and math functions
import random  # for shuffling and sampling
from tqdm import tqdm  # to visualize long-running operations
from collections import Counter  # for counting frequency of elements
import matplotlib.pyplot as plt  # for creating graphs and charts
import seaborn as sns  # enhanced plotting with better aesthetics
from sklearn.model_selection import train_test_split  # for splitting data into train/test sets


In [37]:

# Define the root directory that contains the split folders
# Keep track of the splits you want to load
SPLITS = ["train", "val", "test"]

# Map string labels to integers for model-friendly targets
LABEL_MAP = {"NORMAL": 0, "PNEUMONIA": 1}

def load_split(split_name, target_size=(224, 224), normalize=True):
    """
    Read every image inside ROOT_DIR / split_name / class_label,
    convert it to grayscale, resize, normalize (optional),
    expand the channel dimension, and return the images with labels.
    """
    images = []
    labels = []
    bad_files = []

    # Build the split path
    split_path = os.path.join(ROOT_DIR, split_name)

    for class_name, class_index in LABEL_MAP.items():
        class_path = os.path.join(split_path, class_name)
        if not os.path.exists(class_path):
            continue

        # Iterate through files in the class folder
        for filename in os.listdir(class_path):
            image_path = os.path.join(class_path, filename)

            # Get file extension safely
            _, ext = os.path.splitext(filename)
            if ext.lower() not in {".jpg", ".jpeg", ".png"}:
                continue

            # Read the image directly in grayscale
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                bad_files.append(image_path)
                continue

            # Resize to a fixed target size
            if target_size is not None:
                img = cv2.resize(img, target_size)

            # Normalize pixel values (0–1 range)
            if normalize:
                img = img.astype("float32") / 255.0

            # Expand channel axis to (H, W, 1)
            img = np.expand_dims(img, axis=-1)

            images.append(img)
            labels.append(class_index)

    if bad_files:
        print(f"[WARNING] Skipped {len(bad_files)} corrupted files in '{split_name}'")

    return images, labels


# Load every split using the reusable function
x_train, y_train = load_split("train")
x_val, y_val     = load_split("val")
x_test, y_test   = load_split("test")

# Convert lists to NumPy arrays
x_train, y_train = np.array(x_train), np.array(y_train)
x_val,   y_val   = np.array(x_val),   np.array(y_val)
x_test,  y_test  = np.array(x_test),  np.array(y_test)

# Display the dataset stats
print(f"Train set: {x_train.shape}, labels: {y_train.shape}")
print(f"Validation set: {x_val.shape}, labels: {y_val.shape}")
print(f"Test set: {x_test.shape}, labels: {y_test.shape}")

# Show class balance
print("Class balance in train:", Counter(y_train))
print("Class balance in val:  ", Counter(y_val))
print("Class balance in test: ", Counter(y_test))


Train set: (5216, 224, 224, 1), labels: (5216,)
Validation set: (16, 224, 224, 1), labels: (16,)
Test set: (624, 224, 224, 1), labels: (624,)
Class balance in train: Counter({1: 3875, 0: 1341})
Class balance in val:   Counter({0: 8, 1: 8})
Class balance in test:  Counter({1: 390, 0: 234})


In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# # Convert your existing NumPy arrays to PyTorch tensors
# x_train_tensor = torch.FloatTensor(x_train).permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)
# y_train_tensor = torch.LongTensor(y_train)

# x_val_tensor = torch.FloatTensor(x_val).permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)
# y_val_tensor = torch.LongTensor(y_val)

# x_test_tensor = torch.FloatTensor(x_test).permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)
# y_test_tensor = torch.LongTensor(y_test)

# print(f"Train: {x_train_tensor.shape}, {y_train_tensor.shape}")
# print(f"Val: {x_val_tensor.shape}, {y_val_tensor.shape}")
# print(f"Test: {x_test_tensor.shape}, {y_test_tensor.shape}")

In [39]:
def preprocess_low_res(images, low_res=(40,40), final_size=(224,224)):
    processed = []
    for img in images:
        # img: (H,W,C) numpy
        img = (img*255).astype(np.uint8)
        if img.shape[2] == 1:
            img = img[:,:,0]
        img_low = cv2.resize(img, low_res, interpolation=cv2.INTER_AREA)
        img_up = cv2.resize(img_low, final_size, interpolation=cv2.INTER_CUBIC)
        img_up = np.expand_dims(img_up, axis=-1).astype(np.float32)/255.0
        processed.append(img_up)
    return np.array(processed)

# Apply to all splits once
x_train = preprocess_low_res(x_train)
x_val   = preprocess_low_res(x_val)
x_test  = preprocess_low_res(x_test)

# Convert to tensors and create datasets as before
x_train_tensor = torch.FloatTensor(x_train).permute(0,3,1,2)
x_val_tensor   = torch.FloatTensor(x_val).permute(0,3,1,2)
x_test_tensor  = torch.FloatTensor(x_test).permute(0,3,1,2)


In [40]:
# Alternate to TensorDataset but from scratch :)
class ChestXRayDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    # Required by DataLoader to know when to stop iterating
    def __len__(self):
        return len(self.images)

    # Required by DataLoader to fetch individual samples during batch creation
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        return image, label

# Create datasets
train_dataset = ChestXRayDataset(x_train_tensor, y_train_tensor)
val_dataset = ChestXRayDataset(x_val_tensor, y_val_tensor)
test_dataset = ChestXRayDataset(x_test_tensor, y_test_tensor)

In [41]:
# Calculate class weights for sampler
class_counts = Counter(y_train)
total_samples = len(y_train)
class_weights = {0: total_samples / class_counts[0],  # NORMAL
                 1: total_samples / class_counts[1]}   # PNEUMONIA

# Create sample weights for each training example
sample_weights = [class_weights[label.item()] for label in y_train_tensor]

# Create weighted sampler
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, sampler=weighted_sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [42]:
class PneumoniaCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):  # 1 in channels: Grayscaled images, 2 classes: NORMAL and PNEUMONIA
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),    # 3x3 conv
                nn.BatchNorm2d(out_c),                   # BatchNorm
                nn.ReLU(inplace=True),                   # Relu
                nn.Conv2d(out_c, out_c, 3, padding=1),   # 3x3 conv (more complex features)
                nn.BatchNorm2d(out_c),                   # BatchNorm
                nn.ReLU(inplace=True),                   # Relu
                nn.MaxPool2d(2)                          # MaxPol (reduce dimensions by half)
            )

        self.layer1 = conv_block(in_channels, 32)        # first conv block (1 -> 32)
        self.layer2 = conv_block(32, 64)                 # second conv block (32 -> 64)
        self.layer3 = conv_block(64, 128)                # third conv block (64 -> 128)
        self.layer4 = conv_block(128, 256)               # forth conv block (128 -> 256)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # converts size to (256, 1, 1)
        self.dropout = nn.Dropout(0.4)                   # random 40% dropout
        self.fc = nn.Linear(256, num_classes)            # fully connected layer (256 -> 2)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# Initialize the model
model = PneumoniaCNN(num_classes=2)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model parameters: 1,174,114


In [43]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Handles multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Move data to device
def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

In [44]:
# import torch
# import cv2
# import numpy as np

# LOW_RES = (40, 40)
# FINAL_SIZE = (224, 224)

# def degrade_image_batch(batch, final_size=FINAL_SIZE, low_res=LOW_RES):
#     """
#     Downsamples each image to `low_res` and upsamples back to `final_size`.
#     Works with grayscale images (N,H,W) or (N,C,H,W).
#     """
#     # Ensure batch has 4 dims (N,C,H,W)
#     if batch.dim() == 3:  # (N,H,W)
#         batch = batch.unsqueeze(1)  # -> (N,1,H,W)

#     batch_np = batch.permute(0, 2, 3, 1).cpu().numpy()  # (N,H,W,C)
#     degraded = []

#     for img in batch_np:
#         img = (img * 255).astype(np.uint8)

#         # If single-channel, make sure shape is (H,W)
#         if img.shape[2] == 1:
#             img = img[:, :, 0]

#         img_low = cv2.resize(img, low_res, interpolation=cv2.INTER_AREA)
#         img_up = cv2.resize(img_low, final_size, interpolation=cv2.INTER_CUBIC)

#         # Add channel dimension back if needed
#         if len(img_up.shape) == 2:
#             img_up = np.expand_dims(img_up, axis=-1)

#         img_up = img_up.astype(np.float32) / 255.0
#         degraded.append(img_up)

#     degraded = np.stack(degraded, axis=0)  # (N,H,W,C)
#     # convert back to tensor (N,C,H,W)
#     degraded_tensor = torch.tensor(degraded).permute(0, 3, 1, 2).to(batch.device)
#     return degraded_tensor


In [45]:
# Training loop
def train_model(model, train_loader, val_loader, epochs=20):
    best_val_acc = 0

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = to_device(images, device), to_device(labels, device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = to_device(images, device), to_device(labels, device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        # Calculate metrics
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%')
        print('-' * 50)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_pneumonia_model.pth')

    print(f'Best validation accuracy: {best_val_acc:.2f}%')

# Train the model
train_model(model, train_loader, val_loader, epochs=20)

KeyboardInterrupt: 

In [ ]:
import torch.nn.functional as F

# Model evaluation
def evaluate_model(model, test_loader):
    model.eval()
    model.load_state_dict(torch.load('best_pneumonia_model.pth'))

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = to_device(images, device), to_device(labels, device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probabilities[:, 1].cpu().numpy())  # Pneumonia probability

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    auc = roc_auc_score(all_labels, all_probs)

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall: {recall:.4f}")
    print(f"Test F1-Score: {f1:.4f}")
    print(f"Test AUC: {auc:.4f}")

# Evaluate
evaluate_model(model, test_loader)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def plot_confusion_matrix_simple(model, test_loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    disp = ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=["NORMAL", "PNEUMONIA"], cmap="Blues", values_format="d"

    )
    plt.figtext(0.02, 0.02,
                f'Accuracy: {0.9574:.4f} | Precision: {0.9575:.4f} | '
                f'Recall: {0.9574:.4f} | F1-Score: {0.9575:.4f} | AUC: {0.9913:.4f}',
                ha='left', fontsize=10, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
    disp.ax_.set_title("Confusion Matrix - Pneumonia Detection")
    plt.show()

plot_confusion_matrix_simple(model, test_loader, device)


HERE

In [53]:
import os
import cv2
import numpy as np
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import random
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Paths and Labels
# -----------------------------
ROOT_DIR = "/kaggle/input/chest-xray-pneumonia/chest_xray"
SPLITS = ["train", "val", "test"]
LABEL_MAP = {"NORMAL": 0, "PNEUMONIA": 1}

# -----------------------------
# 2️⃣ Load & Preprocess to 40x40
# -----------------------------
LOW_RES = (20, 20)

def load_and_preprocess(split_name, low_res=LOW_RES):
    images = []
    labels = []
    split_path = os.path.join(ROOT_DIR, split_name)

    for class_name, class_index in LABEL_MAP.items():
        class_path = os.path.join(split_path, class_name)
        if not os.path.exists(class_path):
            continue

        for filename in os.listdir(class_path):
            image_path = os.path.join(class_path, filename)
            if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Resize directly to low resolution
            img = cv2.resize(img, low_res, interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 255.0  # normalize
            img = np.expand_dims(img, axis=0)     # channel-first: (C,H,W)

            images.append(img)
            labels.append(class_index)

    images = np.array(images)
    labels = np.array(labels)
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

# Load all splits
x_train_tensor, y_train_tensor = load_and_preprocess("train")
x_val_tensor, y_val_tensor     = load_and_preprocess("val")
x_test_tensor, y_test_tensor   = load_and_preprocess("test")

print("Train:", x_train_tensor.shape, y_train_tensor.shape)
print("Val:", x_val_tensor.shape, y_val_tensor.shape)
print("Test:", x_test_tensor.shape, y_test_tensor.shape)

# -----------------------------
# 3️⃣ Dataset and DataLoader
# -----------------------------
class ChestXRayDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Weighted sampler for class imbalance
class_counts = Counter(y_train_tensor.numpy())
total_samples = len(y_train_tensor)
class_weights = {0: total_samples/class_counts[0], 1: total_samples/class_counts[1]}
sample_weights = [class_weights[label.item()] for label in y_train_tensor]

train_dataset = ChestXRayDataset(x_train_tensor, y_train_tensor)
val_dataset   = ChestXRayDataset(x_val_tensor, y_val_tensor)
test_dataset  = ChestXRayDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=WeightedRandomSampler(sample_weights, len(sample_weights)), num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# -----------------------------
# 4️⃣ CNN Model
# -----------------------------
class PneumoniaCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )

        self.layer1 = conv_block(in_channels, 32)
        self.layer2 = conv_block(32, 64)
        self.layer3 = conv_block(64, 128)
        self.layer4 = conv_block(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# -----------------------------
# 5️⃣ Training Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PneumoniaCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

# -----------------------------
# 6️⃣ Training Loop
# -----------------------------
def train_model(model, train_loader, val_loader, epochs=20):
    best_val_acc = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0,0,0

        for images, labels in train_loader:
            images, labels = to_device(images, device), to_device(labels, device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs,1)
            train_total += labels.size(0)
            train_correct += (predicted==labels).sum().item()

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0,0,0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = to_device(images, device), to_device(labels, device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs,1)
                val_total += labels.size(0)
                val_correct += (predicted==labels).sum().item()

        train_acc = 100*train_correct/train_total
        val_acc   = 100*val_correct/val_total
        print(f'Epoch [{epoch+1}/{epochs}] Train Loss:{train_loss/len(train_loader):.4f} Train Acc:{train_acc:.2f}% | Val Loss:{val_loss/len(val_loader):.4f} Val Acc:{val_acc:.2f}%')

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),'best_pneumonia_model.pth')

    print(f"Best validation accuracy: {best_val_acc:.2f}%")

# -----------------------------
# 7️⃣ Train
# -----------------------------
train_model(model, train_loader, val_loader, epochs=10)

# -----------------------------
# 8️⃣ Evaluate
# -----------------------------
def evaluate_model(model, test_loader):
    model.eval()
    model.load_state_dict(torch.load('best_pneumonia_model.pth'))

    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = to_device(images, device), to_device(labels, device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs,1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:,1].cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    auc = roc_auc_score(all_labels, all_probs)
    print(f"Test Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

evaluate_model(model, test_loader)


Train: torch.Size([5216, 1, 20, 20]) torch.Size([5216])
Val: torch.Size([16, 1, 20, 20]) torch.Size([16])
Test: torch.Size([624, 1, 20, 20]) torch.Size([624])
Epoch [1/10] Train Loss:0.1812 Train Acc:93.06% | Val Loss:0.7437 Val Acc:68.75%
Epoch [2/10] Train Loss:0.0966 Train Acc:96.49% | Val Loss:1.0311 Val Acc:68.75%
Epoch [3/10] Train Loss:0.0728 Train Acc:97.30% | Val Loss:0.8259 Val Acc:75.00%
Epoch [4/10] Train Loss:0.0463 Train Acc:98.62% | Val Loss:0.2350 Val Acc:81.25%
Epoch [5/10] Train Loss:0.0447 Train Acc:98.52% | Val Loss:0.3614 Val Acc:81.25%
Epoch [6/10] Train Loss:0.0405 Train Acc:98.43% | Val Loss:0.5655 Val Acc:81.25%
Epoch [7/10] Train Loss:0.0335 Train Acc:98.75% | Val Loss:1.3483 Val Acc:68.75%
Epoch [8/10] Train Loss:0.0281 Train Acc:98.89% | Val Loss:0.5853 Val Acc:75.00%
Epoch [9/10] Train Loss:0.0247 Train Acc:99.14% | Val Loss:1.0484 Val Acc:68.75%
Epoch [10/10] Train Loss:0.0260 Train Acc:99.12% | Val Loss:0.2414 Val Acc:81.25%
Best validation accuracy: 81.2

In [54]:
from sklearn.metrics import confusion_matrix

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)


Confusion Matrix:
 [[147  87]
 [ 19 371]]
